# 🎓 Mejora de Imagen, Ruido y Detección de Bordes

En esta sesión nos centraremos en mejorar la calidad de las imágenes para facilitar su procesamiento y en detectar la estructura más importante de los objetos: sus bordes.

### 🎯 Objetivos de Aprendizaje
1.  Entender qué es el **ruido** y cómo eliminarlo (Suavizado).
2.  Conocer los diferentes tipos de filtros (**Gaussiano, Mediana, Bilateral**).
3.  Detectar cambios bruscos de intensidad (**Gradientes y Bordes**).
4.  Implementar el algoritmo de **Canny**.

---

## 1. Suavizado y Reducción de Ruido (Blurring)

El ruido son variaciones aleatorias en el brillo o color. Para reducirlo, "suavizamos" la imagen promediando píxeles vecinos.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Cargar una imagen con ruido (o simularla)
imagen = cv2.imread('moneda.jpg') # Asegúrate de tener una imagen o cámbialo

if imagen is None:
    # Generar imagen sintética con ruido si no existe archivo
    imagen = np.zeros((300, 300, 3), dtype=np.uint8)
    cv2.circle(imagen, (150, 150), 100, (255, 255, 255), -1)
    ruido = np.random.normal(0, 25, imagen.shape).astype(np.uint8)
    imagen = cv2.add(imagen, ruido)

# 1. Filtro de Media (Average Blur): Promedio simple
blur_media = cv2.blur(imagen, (5, 5))

# 2. Filtro Gaussiano: Promedio ponderado (más peso al centro). Más natural.
blur_gauss = cv2.GaussianBlur(imagen, (5, 5), 0)

# 3. Filtro de Mediana: Excelente para ruido "Sal y Pimienta" (puntos blancos/negros).
blur_median = cv2.medianBlur(imagen, 5)

# 4. Filtro Bilateral: Suaviza pero RESPETA los bordes (es más lento).
blur_bilateral = cv2.bilateralFilter(imagen, 9, 75, 75)

# Visualización
plt.figure(figsize=(12, 8))
plt.subplot(2, 3, 1); plt.imshow(imagen); plt.title("Original con Ruido")
plt.subplot(2, 3, 2); plt.imshow(blur_media); plt.title("Media (Average)")
plt.subplot(2, 3, 3); plt.imshow(blur_gauss); plt.title("Gaussiano")
plt.subplot(2, 3, 4); plt.imshow(blur_median); plt.title("Mediana")
plt.subplot(2, 3, 5); plt.imshow(blur_bilateral); plt.title("Bilateral (Bordes)")
plt.tight_layout()
plt.show()

## 2. Detección de Bordes (Canny)

El algoritmo de Canny es el estándar de oro en detección de bordes. Es un proceso multi-etapa:
1.  Reducción de ruido (Gaussiano).
2.  Cálculo de gradientes (intensidad del cambio).
3.  Supresión de no-máximos (afinar líneas).
4.  Umbralización por histéresis (decidir qué es borde y qué no).

In [ ]:
# Cargar imagen y convertir a Gris
img_gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# Aplicar Canny
# Argumentos: (imagen, umbral_minimo, umbral_maximo)
bordes_bajos = cv2.Canny(img_gris, 50, 100)   # Detecta muchos bordes (incluso ruido)
bordes_altos = cv2.Canny(img_gris, 100, 200)  # Detecta solo bordes fuertes

plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1); plt.imshow(img_gris, cmap='gray'); plt.title("Escala de Grises")
plt.subplot(1, 3, 2); plt.imshow(bordes_bajos, cmap='gray'); plt.title("Canny (Umbrales Bajos)")
plt.subplot(1, 3, 3); plt.imshow(bordes_altos, cmap='gray'); plt.title("Canny (Umbrales Altos)")
plt.show()

### 2.1 Canny en Tiempo Real (Webcam)
Prueba esto para ver cómo los bordes cambian con el movimiento.

In [ ]:
cap = cv2.VideoCapture(0)
print("Iniciando Webcam con Canny. Presiona 'q' para salir.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Convertir a gris y aplicar Canny
    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bordes = cv2.Canny(gris, 80, 150)
    
    cv2.imshow('Bordes Canny', bordes)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 3. 🚀 Mini-Proyecto: Restaurador de Fotos Antiguas

**Objetivo:** Crea una función que tome una foto "antigua" (con ruido y poco contraste) y:
1.  Aplique un filtro **Bilateral** para quitar ruido sin borrar detalles.
2.  Mejore el contraste (puedes investigar `cv2.equalizeHist` o CLAHE).
3.  (Opcional) Aplique un ligero afilado (Sharpening).

El código base está listo para que completes la lógica.

In [1]:
# TODO: Implementa el mini-proyecto aquí

import cv2
import numpy as np

# --- 1. Generación de Imagen de Prueba (Simular Foto Antigua) ---
# Creamos un fondo gris
imagen = np.full((400, 600, 3), 200, dtype=np.uint8)
# Añadimos texto que simule detalles que NO queremos borrar
cv2.putText(imagen, "RECUERDO 1980", (80, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (100, 100, 100), 4)
cv2.circle(imagen, (500, 300), 50, (100, 100, 100), -1)

# Añadir Ruido Gaussiano (Grano de película)
ruido = np.random.normal(0, 25, imagen.shape).astype(np.uint8)
imagen_ruidosa = cv2.subtract(imagen, ruido) # Ensuciar la imagen

# Reducir el contraste (hacerla ver "lavada" o vieja)
imagen_ruidosa = (imagen_ruidosa * 0.7).astype(np.uint8)

print("⏳ Procesando imagen...")

# --- 2. Pipeline de Restauración ---

# PASO A: Reducción de Ruido Inteligente
# Usamos Filtro Bilateral porque suaviza el "grano" pero RESPETA los bordes del texto.
# Un blur normal (Gaussiano) dejaría el texto borroso.
# Argumentos: (img, d, sigmaColor, sigmaSpace)
imagen_limpia = cv2.bilateralFilter(imagen_ruidosa, 9, 75, 75)

# PASO B: Mejora de Contraste (CLAHE)
# Para mejorar el contraste sin saturar los colores, trabajamos en el espacio LAB.
# LAB separa la Luminosidad (L) del color (A, B).

# 1. Convertir a LAB
lab = cv2.cvtColor(imagen_limpia, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)

# 2. Aplicar CLAHE (Contrast Limited Adaptive Histogram Equalization) al canal L
# Esto ecualiza la luz localmente, resaltando detalles ocultos.
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
l_mejorado = clahe.apply(l)

# 3. Unir canales y volver a BGR
lab_mejorado = cv2.merge((l_mejorado, a, b))
imagen_restaurada = cv2.cvtColor(lab_mejorado, cv2.COLOR_LAB2BGR)

# --- 3. Visualización ---

# Concatenar imágenes horizontalmente para comparar
comparativa = np.hstack((imagen_ruidosa, imagen_restaurada))

cv2.imshow("Izquierda: Original | Derecha: Restaurada", comparativa)

print("✅ ¡Foto restaurada!")
print("Nota: Observa cómo el fondo es más suave pero las letras siguen nítidas.")
print("Presiona cualquier tecla para cerrar...")

cv2.waitKey(0)
cv2.destroyAllWindows()

⏳ Procesando imagen...
✅ ¡Foto restaurada!
Nota: Observa cómo el fondo es más suave pero las letras siguen nítidas.
Presiona cualquier tecla para cerrar...
